# 足球比赛结果预测（竞技分析场景）

**EDP V2.0 应用示例 · 体育数据科学 / 竞技实力建模**

本 notebook 演示如何用 EDP 框架，结合 **Glicko-2 动态评级** 与多源竞技信号，
对一场足球比赛的三种可能结果（主胜 / 平 / 客胜）进行**概率态势感知**。

---

## ⚠️ 用途边界声明

本示例为**体育数据科学与竞技分析研究用途**，演示概率推断与动态评级方法本身。

- 输出的比赛结果概率是**统计研究估计**，不是投注建议；
- **不构成**对任何博彩、竞猜、彩票、金融衍生品的参与建议；
- 本示例**不使用**市场赔率、盘口、水位等任何博彩市场信号，仅使用
  竞技层面的客观信号（队伍评级、近期状态、交锋记录等）；
- 足球比赛结果具有高度随机性，模型预测不保证命中；
- 在任何博彩受监管的地区，参与博彩须遵守当地法律并自负风险。

使用者须自行承担一切决策风险。

---

## 场景设定

**研究对象**：一场假设的联赛比赛 —— 主队 `Home FC` vs 客队 `Away United`。

**可能结果（全域）**：
1. `home_win` —— 主队胜
2. `draw`     —— 平局
3. `away_win` —— 客队胜

**信号来源（多源竞技证据，不含博彩市场信号）**：
- Glicko-2 评级模型（基于历史比赛结果动态更新）
- 近期状态（近 N 场战绩）
- 历史交锋（H2H）
- 主场优势
- 伤停情况（关键球员缺阵影响）

## 1. 环境与框架加载

In [ ]:
import os, sys, importlib.util

# 加载 EDP 包（仓库内 src/python）
_SRC = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src', 'python'))
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

_spec = importlib.util.spec_from_file_location(
    'edp', os.path.join(_SRC, '__init__.py'), submodule_search_locations=[_SRC]
)
_edp = importlib.util.module_from_spec(_spec)
sys.modules['edp'] = _edp
_spec.loader.exec_module(_edp)

from edp import (
    EDP, GenericDomain, Outcome, Evidence,
    ProbabilityEngine, ConformalEngine, ConformalConfig,
)
print('EDP 版本:', _edp.__version__)

## 2. Glicko-2 动态评级：从历史比赛结果建模队伍实力

Glicko-2 是 Elo 评级的改进版，引入评级偏差（RD）与波动率（σ），
对比赛样本较少或长期未参赛的队伍给出更稳健的实力估计。

> 注：以下历史比赛数据为**演示用合成数据**，不代表任何真实球队。

In [ ]:
engine = ProbabilityEngine()

# 合成的近期历史比赛序列：(主队, 客队, 主队实际得分: 1胜/0.5平/0负)
# 每场比赛后用 Glicko-2 增量更新两队评级
match_history = [
    ('Home FC',  'Away United', 1.0),  # Home 胜
    ('Home FC',  'Rivals SC',   0.5),  # 平
    ('Away United','Rivals SC', 1.0),  # Away 胜
    ('Home FC',  'City FC',     1.0),  # Home 胜
    ('Away United','City FC',   0.0),  # Away 负
    ('Home FC',  'Town FC',     1.0),  # Home 胜
    ('Away United','Town FC',   1.0),  # Away 胜
    ('Home FC',  'Away United', 0.5),  # 平（直接交锋）
    ('Home FC',  'Rivals SC',   1.0),  # Home 胜
    ('Away United','Rivals SC', 1.0),  # Away 胜
]

for home, away, score in match_history:
    h = engine.get_or_create_rating(home)
    a = engine.get_or_create_rating(away)
    # 双向更新：主队以实际得分，客队以对称得分
    engine.update_glicko_rating(home, [(score, a.rating, a.rd)])
    engine.update_glicko_rating(away, [(1.0 - score, h.rating, h.rd)])

for tid in ('Home FC', 'Away United'):
    r = engine.get_or_create_rating(tid)
    print(f"{tid:14s}  评级={r.rating:7.1f}  RD={r.rd:6.1f}  σ={r.volatility:.3f}  场次={r.games_played}")

## 3. 由评级生成模型先验

用 Glicko-2 的 `predict_with_rating` 得到三结果（主胜/平/客胜）的先验概率，
该函数已内置主场优势项（home_advantage，评级单位）。

In [ ]:
rating_prior = engine.predict_with_rating('Home FC', 'Away United', home_advantage=100.0)
print('Glicko-2 模型先验:')
for oid, p in rating_prior.items():
    print(f'  {oid:10s}: {p:.1%}')

## 4. 定义问题域并注入先验

In [ ]:
domain = GenericDomain([
    Outcome('home_win', '主队胜'),
    Outcome('draw',     '平局'),
    Outcome('away_win', '客队胜'),
])
domain.graph_type = 'chain'  # 三结果有序链

edp = EDP(domain, {
    'conformal': {'method': 'aci', 'alpha': 0.1},  # L7 保形：目标 90% 覆盖
    'allocation': {'kelly_fraction': 0.1},
})
edp.initialize()

# 将 Glicko-2 模型先验作为初始概率
edp.probabilities = dict(rating_prior)
edp._normalize_probabilities()
print('注入后先验:', edp.probabilities)

## 5. 构造多源竞技证据

每条证据指向一个具体结果（定向证据），附来源可靠性与自报信心。

> 注：以下数值为**演示用合成数据**，不代表任何真实赛事信息。

In [ ]:
evidence = [
    # 近期状态：Home 近 5 场 4胜1平 → 倾向 home_win
    Evidence('home_form', 'model',
             {'probability': 0.62}, outcome_id='home_win',
             confidence=0.75, reliability=0.75),

    # 近期状态：Away 近 5 场 3胜2负，状态起伏 → 倾向 away_win（较弱）
    Evidence('away_form', 'model',
             {'probability': 0.45}, outcome_id='away_win',
             confidence=0.55, reliability=0.65),

    # 历史交锋 H2H：近 6 次交锋 Home 3胜2平1负 → 倾向 home_win
    Evidence('h2h', 'model',
             {'probability': 0.58}, outcome_id='home_win',
             confidence=0.65, reliability=0.7),

    # 主场优势（独立于评级中已计入的 100 分项）：略增 home_win
    Evidence('home_adv', 'expert',
             {'probability': 0.55}, outcome_id='home_win',
             confidence=0.6, reliability=0.6),

    # 伤停：Home 一名主力中场伤疑 → 风险信号，倾向 draw
    Evidence('home_injury', 'expert',
             {'probability': 0.40}, outcome_id='draw',
             confidence=0.55, reliability=0.6),
]
print(f'共 {len(evidence)} 条证据，来源类型:',
      sorted({e.source_type for e in evidence}))

## 6. 一键分析：L0 → L7 完整流程

`budget=0`：本示例为竞技分析研究，不涉及任何资金分配。

In [ ]:
# 由于已手动注入先验，这里直接调用证据融合 + 保形预测
assessment = edp.add_evidence(evidence)
edp.snapshot('after_evidence')
pset = edp.conformal_predict()

print('融合后概率:')
for oid, p in sorted(edp.probabilities.items(), key=lambda x: -x[1]):
    print(f'  {oid:10s}: {p:.1%}')
print()
print(f"保形预测集 (目标覆盖率 ≥ {pset.coverage_target:.0%}): {pset.prediction_set}")

## 7. 态势评估：共识、稳定性、模型多样性

In [ ]:
print(f"共识度:      {assessment.consensus_score:.3f}")
print(f"稳定性分类:  {assessment.stability.value}")
print(f"融合方法:    {assessment.fusion_method}")
print(f"置信度:      {assessment.confidence:.3f}")
if assessment.anomaly_flags:
    print(f"异常来源:    {assessment.anomaly_flags}")
else:
    print('异常来源:    无')

# 模型多样性（DTVW）：判断证据是否冗余
from edp import DomainAwarenessEngine, EvidenceSource, EvidenceType, SourceReliability
from datetime import datetime
sources = [
    EvidenceSource(e.id, EvidenceType(e.source_type),
                   SourceReliability.C, datetime.now(),
                   {'probability': e.extract_probability()}, e.confidence)
    for e in evidence
]
div = DomainAwarenessEngine.model_diversity(sources)
print()
print('模型多样性分析:')
print(f"  多样性:        {div['diversity']:.3f}")
print(f"  冗余度:        {div['redundancy']:.3f}")
print(f"  有效来源数:    {div['effective_sources']:.2f} / {len(sources)}")

## 8. 保形预测集解读（L7：有限样本覆盖率保证）

保形预测输出一个**预测集**，在有限样本下以 ≥ 90% 的概率包含真实结果。
ACI 模式对分布漂移（如赛季间实力变化）鲁棒。

In [ ]:
print(f"预测集:        {pset.prediction_set}")
print(f"目标覆盖率:    {pset.coverage_target:.0%}")
print(f"方法:          {pset.method}")
print()
if pset.is_full:
    print('→ 预测集包含全部结果：当前竞技信号不足以窄化判断，建议补充关键证据（如最新伤停）。')
elif len(pset.prediction_set) == 1:
    print('→ 预测集仅含一个结果：证据较强地指向该结果（仍非确定）。')
else:
    print('→ 预测集已收窄：在 90% 覆盖保证下排除了部分结果。')

## 9. 在线校准：随赛季结果迭代

随着比赛结果陆续揭晓，可用 `conformal_update` 在线更新保形层，
使未来预测集的覆盖率在分布漂移下仍收敛到目标。

In [ ]:
# 模拟：对历史若干场比赛的预测概率与实际结果，在线更新保形层
historical = [
    ({'home_win':0.5,'draw':0.3,'away_win':0.2}, 'home_win'),
    ({'home_win':0.4,'draw':0.3,'away_win':0.3}, 'draw'),
    ({'home_win':0.3,'draw':0.3,'away_win':0.4}, 'away_win'),
    ({'home_win':0.5,'draw':0.3,'away_win':0.2}, 'home_win'),
    ({'home_win':0.4,'draw':0.4,'away_win':0.2}, 'draw'),
]
for preds, actual in historical:
    edp.initialize()
    edp.probabilities = preds
    res = edp.conformal_update(actual)
    print(f"预测={preds} 实际={actual:9s} 覆盖={res['covered']} "
          f"滚动覆盖率={res['coverage_rate']:.2f}")

stats = edp.conformal_engine.coverage_stats()
print(f"\n最终: 经验覆盖率={stats['empirical_coverage']:.2f} "
      f"目标={stats['target_coverage']:.2f} 校准样本={stats['calibration_size']}")

## 10. 结论与边界

**本示例展示了**：
- 用 EDP 把「比赛结果预测」分解为「结果 + 多源竞技信号」，统一融合；
- Glicko-2 评级从历史比赛结果动态建模队伍实力，生成模型先验；
- 定向证据（指向具体结果）避免概率被无差别拉平；
- L7 保形预测集给出有覆盖率保证的结论范围，而非单一确定性断言；
- 模型多样性指标揭示证据冗余度，提示是否需补充独立信号。

**再次声明**：本示例为体育数据科学研究演示，合成数据，**不构成任何投注、博彩、
彩票或金融建议**。足球比赛结果具有高度随机性，模型预测不保证命中。
在博彩受监管的地区，参与博彩须遵守当地法律并自负风险。